# Iteracja po wierszach: `iterrows()` vs `itertuples()` vs Polars `iter_rows()`

**Problem:** iteracja po wierszach DataFrame jest częstym antywzorcem w pandas — w większości przypadków da się ją zastąpić wektoryzacją, ale gdy iteracja jest konieczna (np. wywołania API per wiersz, złożona logika warunkowa niemożliwa do zwektoryzowania), wybór metody ma realny wpływ na wydajność i typowanie danych.

**Porównanie:**
- `iterrows()` — zwraca `(index, Series)`; wolne, bo kopiuje wartości wiersza do `pd.Series` (możliwa utrata/ujednolicenie typów), ale wygodne przy nazwach kolumn ze spacjami.
- `itertuples()` — zwraca namedtuple; szybsze niż `iterrows()`, lepiej zachowuje typy danych; nazwy kolumn ze spacjami wymagają dostępu przez `getattr()`.
- Polars `iter_rows()` — odpowiednik w Polars, zwraca krotkę (domyślnie) lub słownik (`named=True`).

**Kiedy stosować:** iteracja tylko wtedy, gdy operacji nie da się zwektoryzować. W pierwszej kolejności rozważ wektoryzację, maskowanie boolowskie lub `.apply()`.

**Benchmark:** patrz ostatnia sekcja notatnika — porównanie czasowe na 100 000 wierszy.

## Setup

Mały, syntetyczny DataFrame używany we wszystkich przykładach poniżej.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "region": ["North", "South", "Central", "East", "West"],
    "employee_count": [120, 180, 150, 95, 130],
    "account_balance": [15230.50, 22890.10, 18760.00, 12500.75, 19340.20],
    "check_date": pd.to_datetime(
        ["2026-01-15", "2026-01-16", "2026-01-17", "2026-01-18", "2026-01-19"]
    ),
})

df

## `itertuples()` — preferowany sposób iteracji

Zwraca każdy wiersz jako namedtuple. Domyślnie pierwszym polem jest indeks wiersza (`Index`); `index=False` go pomija.

In [ ]:
for row in df.itertuples():
    print(f"W dniu {row.check_date.date()} stan konta wynosił {row.account_balance:.2f}")

print()

# Przykład z warunkiem (index=False pomija indeks wiersza w namedtuple)
for row in df.itertuples(index=False):
    if row.employee_count > 150:
        print(f"{row.region}: {row.employee_count} pracowników (powyżej progu 150)")

## `iterrows()` — zwraca `(index, Series)`

Wolniejsze niż `itertuples()`, ale wygodne, gdy potrzebny jest dostęp przez `row["nazwa_kolumny"]` (np. nazwy kolumn ze spacjami).

In [ ]:
# Podstawowa składnia
for index, row in df.iterrows():
    pass  # index: pozycja wiersza, row: pd.Series z wartościami wiersza

# Jeżeli indeks nie jest potrzebny
for _, row in df.iterrows():
    pass

# Dostęp do pojedynczej kolumny
for _, row in df.iterrows():
    print(row["region"])

print()

# Wykorzystanie kilku kolumn
for _, row in df.iterrows():
    print(f"{row['region']}: {row['employee_count']} pracowników")

## Polars `iter_rows()` — odpowiednik w Polars

Wymaga zainstalowanego pakietu `polars` (`pip install polars`). Domyślnie zwraca krotkę; `named=True` zwraca słownik.

In [ ]:
import polars as pl

df_pl = pl.from_pandas(df)

# Domyślnie zwracana jest krotka (tuple)
for row in df_pl.iter_rows():
    print(row)

print()

# Rozpakowanie wartości pozycyjnie
for region, employee_count, account_balance, check_date in df_pl.iter_rows():
    print(f"{region}: {employee_count}")

print()

# Z nazwami kolumn (named=True) - zwraca słownik
for row in df_pl.iter_rows(named=True):
    print(f"{row['region']}: {row['employee_count']}")

## Benchmark wydajności

Porównanie czasu wykonania na DataFrame ze 100 000 wierszy: `iterrows()` vs `itertuples()` vs wektoryzacja.

In [ ]:
import time
import numpy as np

n = 100_000
df_large = pd.DataFrame({
    "region": np.random.choice(["North", "South", "Central", "East", "West"], n),
    "employee_count": np.random.randint(50, 300, n),
})


def time_it(func, label):
    start = time.perf_counter()
    func()
    elapsed = time.perf_counter() - start
    print(f"{label}: {elapsed:.4f}s")


def run_iterrows():
    total = 0
    for _, row in df_large.iterrows():
        total += row["employee_count"]
    return total


def run_itertuples():
    total = 0
    for row in df_large.itertuples():
        total += row.employee_count
    return total


def run_vectorized():
    return df_large["employee_count"].sum()


time_it(run_iterrows, "iterrows()")
time_it(run_itertuples, "itertuples()")
time_it(run_vectorized, "wektoryzacja (.sum())")

## Podsumowanie

| Metoda | Szybkość | Zachowanie typów | Kiedy używać |
|---|---|---|---|
| `iterrows()` | Najwolniejsza | Konwertuje wiersz do `Series` (możliwa utrata/ujednolicenie typów) | Gdy nazwy kolumn zawierają spacje lub potrzebny jest wygodny dostęp przez `row["nazwa"]` |
| `itertuples()` | Szybsza niż `iterrows()` | Zachowuje typy (namedtuple) | Domyślny wybór, gdy iteracja jest konieczna |
| Polars `iter_rows()` | Zbliżona do `itertuples()`, w ekosystemie Polars | Zachowuje typy Polars | Gdy dane są już w Polars zamiast pandas |
| Wektoryzacja / `.apply()` | Najszybsza | Pełne zachowanie typów | Zawsze pierwszy wybór — iteracja to ostateczność |

**Wniosek:** różnica rośnie wraz z rozmiarem danych — przy analizach na dużych zbiorach (setki tysięcy wierszy) wybór metody iteracji może oznaczać różnicę rzędu kilku–kilkunastu sekund per operacja, a wektoryzacja pozostaje o rząd wielkości szybsza od obu form iteracji.